# Lab 3 – GraphX / GraphFrames

**Yêu cầu trước khi chạy:**
1. Docker đang chạy (`docker compose up -d`)
2. Đã chạy xong `lab03_data_setup.ipynb` (để có data trong Kafka)
3. Virtual env kích hoạt, cài đặt:
   ```bash
   pip install graphframes-py==0.11.0
   ```

**Bài tập:**
- Exercise 0: Đọc dữ liệu từ Kafka (movies, ratings, tags)
- Exercise 1: Popularity Bias – in-degree & weighted in-degree
- Exercise 2: PageRank – top 20 phim relevant nhất
- Exercise 3 (Bonus): Motif – top 10 phim gây tranh cãi nhất

In [2]:
import os, sys
from pyspark.sql import SparkSession

KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

# GraphFrames 0.11.0 – phiên bản mới nhất, tương thích Spark 4.x + Scala 2.13
# Tham khảo: https://graphframes.io/02-quick-start/01-installation.html
packages = (
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
    "org.apache.kafka:kafka-clients:3.6.0,"
    "io.graphframes:graphframes-spark4_2.13:0.11.0"
)

spark = (SparkSession.builder
    .appName("Lab3_GraphX")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", packages)
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# Checkpoint directory – bắt buộc cho GraphFrames connectedComponents
checkpoint_dir = os.path.join(os.getcwd(), "graphx_checkpoint")
os.makedirs(checkpoint_dir, exist_ok=True)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

print(f"✅ Spark {spark.version} + GraphFrames 0.11.0 khởi tạo thành công!")

✅ Spark 4.0.1 + GraphFrames 0.11.0 khởi tạo thành công!


## Exercise 0: Chuẩn bị Movie Data

Đọc dữ liệu từ 3 Kafka topics:
- `Lab1_movies` – thông tin phim
- `Lab1_ratings` – điểm đánh giá
- `Lab1_tags` – nhãn tag

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, LongType, DoubleType
from pyspark.sql.functions import col, from_json

# --- Định nghĩa schema ---
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title",   StringType(),  True),
    StructField("genres",  StringType(),  True),
])

ratings_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("rating",    DoubleType(),  True),
    StructField("timestamp", LongType(),    True),
])

tags_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("tag",       StringType(),  True),
    StructField("timestamp", LongType(),    True),
])

# --- Hàm đọc dữ liệu batch từ Kafka ---
def read_static_kafka(topic, schema):
    return (
        spark.read.format("kafka")
            .option("kafka.bootstrap.servers", KAFKA_BROKERS)
            .option("subscribe", topic)
            .option("startingOffsets", "earliest")
            .load()
            .selectExpr("CAST(value AS STRING) as json_str")
            .select(from_json(col("json_str"), schema).alias("data"))
            .select("data.*")
    )

# --- Đọc 3 topics ---
df_movies  = read_static_kafka("Lab1_movies",  movies_schema)
df_ratings = read_static_kafka("Lab1_ratings", ratings_schema)
df_tags    = read_static_kafka("Lab1_tags",    tags_schema)

# Cache để tránh re-read nhiều lần
df_movies.cache()
df_ratings.cache()
df_tags.cache()

print(f"✅ Đã tải: {df_movies.count():,} phim | {df_ratings.count():,} ratings | {df_tags.count():,} tags")
df_movies.show(3, truncate=False)
df_ratings.show(3)
df_tags.show(3)

✅ Đã tải: 19,484 phim | 201,672 ratings | 7,366 tags
+-------+-----------------------+-------------------------------------------+
|movieId|title                  |genres                                     |
+-------+-----------------------+-------------------------------------------+
|1      |Toy Story (1995)       |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)         |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)|Comedy|Romance                             |
+-------+-----------------------+-------------------------------------------+
only showing top 3 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+-----------

## Xây Dựng Đồ Thị (User → Movie)

Tạo GraphFrame bipartite gồm:
- **Vertices (đỉnh):** Users + Movies
- **Edges (cạnh):** Lượt đánh giá (User → Movie) với trọng số là điểm rating

In [4]:
from graphframes import GraphFrame
from pyspark.sql.functions import lit, desc, col, sum as _sum, abs as _abs

# --- Vertices ---
# Movies: id = movieId (string), type = "movie"
movies_v = (df_movies
    .select(col("movieId").cast("string").alias("id"),
            col("title"),
            lit("movie").alias("type")))

# Users: id = userId (string), type = "user"
users_v = (df_ratings
    .select(col("userId").cast("string").alias("id"))
    .distinct()
    .withColumn("title", lit("User"))
    .withColumn("type",  lit("user")))

vertices = movies_v.unionByName(users_v)

# --- Edges ---
# User → Movie với weight = rating
edges = (df_ratings
    .select(col("userId").cast("string").alias("src"),
            col("movieId").cast("string").alias("dst"),
            col("rating").alias("weight")))

# --- Tạo GraphFrame ---
g = GraphFrame(vertices, edges)

print(f"Graph: {g.vertices.count():,} đỉnh | {g.edges.count():,} cạnh")
print(f"  - Movies: {movies_v.count():,}")
print(f"  - Users : {users_v.count():,}")

Graph: 20,094 đỉnh | 201,672 cạnh
  - Movies: 19,484
  - Users : 610


## Exercise 1: Kiểm Tra Popularity Bias

Tính:
1. **In-degree** = số lượng người đánh giá riêng biệt (num_raters)
2. **Weighted in-degree** = tổng điểm đánh giá (total_rating_score)

Output: Top-20 phim theo in-degree và weighted in-degree.

In [6]:
# 1. In-degree: số rater (= số cạnh đi vào mỗi movie vertex)
in_degree_df = g.inDegrees.withColumnRenamed("inDegree", "num_raters")

# 2. Weighted In-degree: tổng điểm rating
weighted_in_degree_df = (g.edges
    .groupBy("dst")
    .agg(_sum("weight").alias("total_rating_score")))

# Join và lấy Top 20 movies
popularity_df = (movies_v
    .join(in_degree_df,         movies_v.id == in_degree_df.id,         "inner")
    .join(weighted_in_degree_df, movies_v.id == weighted_in_degree_df.dst, "inner")
    .select(movies_v.id, "title", "num_raters", "total_rating_score")
    .orderBy(desc("num_raters"), desc("total_rating_score"))
    .limit(20))

print("--- EXERCISE 1: Top 20 Phim Phổ Biến Nhất (Popularity Bias) ---")
popularity_df.show(truncate=False)

"""
Insights (Exercise 1):
1. Sự phổ biến không đồng nghĩa với chất lượng – các phim có in-degree cao nhất thường
   là blockbuster đại chúng thu hút mọi tầng lớp khán giả, dù điểm trung bình chưa cao.
2. Chênh lệch giữa num_raters và total_rating_score phản ánh mức độ hài lòng trung bình:
   phim có nhiều rater nhưng tổng điểm thấp tương đối là phim bị đánh giá dưới trung bình.
3. Popularity bias có thể làm lu mờ các tác phẩm indie hoặc phim cult – ít người xem
   nhưng nhận điểm tuyệt đối – khiến hệ thống gợi ý tiếp tục thiên vị phim nổi tiếng.
"""

--- EXERCISE 1: Top 20 Phim Phổ Biến Nhất (Popularity Bias) ---
+----+-----------------------------------------+----------+------------------+
|id  |title                                    |num_raters|total_rating_score|
+----+-----------------------------------------+----------+------------------+
|356 |Forrest Gump (1994)                      |658       |2740.0            |
|356 |Forrest Gump (1994)                      |658       |2740.0            |
|318 |Shawshank Redemption, The (1994)         |634       |2808.0            |
|318 |Shawshank Redemption, The (1994)         |634       |2808.0            |
|296 |Pulp Fiction (1994)                      |614       |2577.0            |
|296 |Pulp Fiction (1994)                      |614       |2577.0            |
|593 |Silence of the Lambs, The (1991)         |558       |2322.0            |
|593 |Silence of the Lambs, The (1991)         |558       |2322.0            |
|2571|Matrix, The (1999)                       |556       |2331.0  

'\nInsights (Exercise 1):\n1. Sự phổ biến không đồng nghĩa với chất lượng – các phim có in-degree cao nhất thường\n   là blockbuster đại chúng thu hút mọi tầng lớp khán giả, dù điểm trung bình chưa cao.\n2. Chênh lệch giữa num_raters và total_rating_score phản ánh mức độ hài lòng trung bình:\n   phim có nhiều rater nhưng tổng điểm thấp tương đối là phim bị đánh giá dưới trung bình.\n3. Popularity bias có thể làm lu mờ các tác phẩm indie hoặc phim cult – ít người xem\n   nhưng nhận điểm tuyệt đối – khiến hệ thống gợi ý tiếp tục thiên vị phim nổi tiếng.\n'

## Exercise 2: Top 20 Phim Relevant Nhất (PageRank)

Chạy **PageRank toàn cục** trên đồ thị User → Movie để tìm các phim được nhiều người xem hệ thống đánh giá cao.

In [7]:
print("⏳ Đang chạy PageRank (khoảng 1-2 phút)...")

pr_results = g.pageRank(resetProbability=0.15, maxIter=5)

# Lấy genres từ df_movies gốc
movies_with_genres = df_movies.select(
    col("movieId").cast("string").alias("id"),
    "genres"
)

# Lọc chỉ lấy movie vertices, join genres, Top 20
top_20_pagerank = (pr_results.vertices
    .filter(col("type") == "movie")
    .join(movies_with_genres, "id")
    .select("id", "title", "genres", "pagerank")
    .orderBy(desc("pagerank"))
    .limit(20))

print("--- EXERCISE 2: Top 20 Phim Relevant Nhất (PageRank) ---")
top_20_pagerank.show(truncate=False)

⏳ Đang chạy PageRank (khoảng 1-2 phút)...


26/05/12 00:32:31 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/12 00:32:31 WARN ShippableVertexPartitionOps: Joining two VertexPartitions with different indexes is slow.
26/05/12 00:32:47 WARN PageRank: Returned DataFrame is persistent and materialized!


--- EXERCISE 2: Top 20 Phim Relevant Nhất (PageRank) ---
+---+--------------------------------+------------------------+-----------------+
|id |title                           |genres                  |pagerank         |
+---+--------------------------------+------------------------+-----------------+
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Crime|Drama             |8.085212329014052|
|318|Shawshank Redemption, The (1994)|Cri

## Exercise 3 (Bonus): Motif – Phim Gây Tranh Cãi Nhất

Tìm các cặp người dùng đánh giá cùng một phim với **chênh lệch tuyệt đối ≥ 3.0**.

Output: Top 10 phim có nhiều cặp phân cực nhất, kèm `movieId`, `title`, `polarized_pairs`.

In [8]:
# Motif: tìm pattern (u1)-[e1]->(m)<-[e2]-(u2)
# tức là 2 user khác nhau đều rate cùng 1 phim
motifs = (g.find("(u1)-[e1]->(m); (u2)-[e2]->(m)")
    .filter("u1.id < u2.id")                                    # Tránh đếm trùng (a,b) và (b,a)
    .filter(col("u1.type") == "user")                           # Đảm bảo u1 là user
    .filter(col("u2.type") == "user")                           # Đảm bảo u2 là user
    .filter(col("m.type")  == "movie")                          # Đảm bảo m là movie
    .filter(_abs(col("e1.weight") - col("e2.weight")) >= 3.0))  # Chênh lệch rating >= 3.0

# Gom nhóm theo phim, đếm số cặp phân cực, lấy movieId từ vertex id
polarization_df = (motifs
    .groupBy(col("m.id").alias("movieId"), col("m.title").alias("title"))
    .count()
    .withColumnRenamed("count", "polarized_pairs")
    .orderBy(desc("polarized_pairs"))
    .limit(10))

print("--- EXERCISE 3 (BONUS): Top 10 Phim Gây Tranh Cãi Nhất ---")
polarization_df.show(truncate=False)

"""
Insights (Exercise 3):
1. Các phim lọt top phân cực thường có cốt truyện phi tuyến tính, kết thúc mở hoặc
   chủ đề gây tranh cãi (chính trị, tôn giáo), khiến khán giả chia hai thái cực rõ rệt.
2. Số polarized_pairs cao cảnh báo hệ thống gợi ý cần thận trọng hơn – đề xuất sai
   đối tượng có thể dẫn đến trải nghiệm tiêu cực và giảm retention người dùng.
3. Những phim có tính phân cực mạnh thường có mức lan truyền và tranh luận trên mạng
   xã hội cao hơn hẳn so với phim giải trí thuần túy – đây là dấu hiệu viral tốt cho
   chiến lược marketing.
"""

--- EXERCISE 3 (BONUS): Top 10 Phim Gây Tranh Cãi Nhất ---


[Stage 336:>                                                        (0 + 1) / 1]

+-------+---------------------------------------------------------+---------------+
|movieId|title                                                    |polarized_pairs|
+-------+---------------------------------------------------------+---------------+
|296    |Pulp Fiction (1994)                                      |24664          |
|2571   |Matrix, The (1999)                                       |21064          |
|527    |Schindler's List (1993)                                  |13976          |
|356    |Forrest Gump (1994)                                      |12552          |
|110    |Braveheart (1995)                                        |12440          |
|2858   |American Beauty (1999)                                   |11992          |
|260    |Star Wars: Episode IV - A New Hope (1977)                |11880          |
|593    |Silence of the Lambs, The (1991)                         |11216          |
|4993   |Lord of the Rings: The Fellowship of the Ring, The (2001)|9424     

'\nInsights (Exercise 3):\n1. Các phim lọt top phân cực thường có cốt truyện phi tuyến tính, kết thúc mở hoặc\n   chủ đề gây tranh cãi (chính trị, tôn giáo), khiến khán giả chia hai thái cực rõ rệt.\n2. Số polarized_pairs cao cảnh báo hệ thống gợi ý cần thận trọng hơn – đề xuất sai\n   đối tượng có thể dẫn đến trải nghiệm tiêu cực và giảm retention người dùng.\n3. Những phim có tính phân cực mạnh thường có mức lan truyền và tranh luận trên mạng\n   xã hội cao hơn hẳn so với phim giải trí thuần túy – đây là dấu hiệu viral tốt cho\n   chiến lược marketing.\n'